# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimahmahmood/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
*One row = one what, over which dates? State it, then verify it below.*
One row shows one content item for one client on one day.

I will use March 2026 as my development month. I will use information that was available before the prediction to identify whether a content item is declining.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────────┐
│ report_date │ client_hash_id │ content_hash_id │ row_count │
│    date     │    varchar     │     varchar     │   int64   │
├─────────────┴────────────────┴─────────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** gsc_impressions:  how many times the page appeared in Google Search.
gsc_clicks: how many times people clicked the page from Google.

ga4_pageviews: how many times the page was viewed.

ga4_sessions: how many visits the page received.

ga4_engaged_sessions: how many visits involved engagement.

These features are available before the prediction.

**Label / proxy:** I will use future Google Search impressions to measure whether a page's performance declines.

**Context:** client_hash_id, content_hash_id, and report_date help identify and organize the data. They will not be used as features.

**Excluded:** I will not use future information when creating features. I will also exclude information that is used to create the label because that would give the model the answer.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [14]:
con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         413966 │
└────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
One limitation is that some clients have less historical data than others so the results may not be the same for every client.

The data can show that a page is declining but it cannot tell us exactly why. The reason could be a content change, Google update, competition, seasonality, or something else.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [6]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [7]:
%pip install -q duckdb huggingface_hub pandas

In [8]:
import duckdb

con = duckdb.connect()

print("DuckDB connected")

DuckDB connected


In [9]:
import os

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [os.environ["HF_TOKEN"]])

print("Hugging Face access configured")

Hugging Face access configured


In [10]:
result = con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │ first_date │ last_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

In [18]:
march_april = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    ga4_pageviews,
    ga4_sessions,
    ga4_engaged_sessions,
    gsc_data_available,
    ga4_data_available
FROM read_parquet([
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet',
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
])
""")

march_april

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬───────────────┬──────────────┬──────────────────────┬────────────────────┬────────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ ga4_pageviews │ ga4_sessions │ ga4_engaged_sessions │ gsc_data_available │ ga4_data_available │
│         varchar         │         varchar          │    date     │      int64      │   int64    │     int64     │    int64     │        int64         │      boolean       │      boolean       │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼───────────────┼──────────────┼──────────────────────┼────────────────────┼────────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │          NULL │         NULL │                 NULL │ true               │ NULL               │
│ client_73cda7b4e4f

In [21]:
labeled_data = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    march.client_hash_id,
    march.content_hash_id,
    march.march_impressions,
    april.april_impressions,

    CASE
        WHEN april.april_impressions < march.march_impressions * 0.8
        THEN 1
        ELSE 0
    END AS declining_label

FROM march
INNER JOIN april
USING (client_hash_id, content_hash_id)
""")

labeled_data

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬───────────────────┬───────────────────┬─────────────────┐
│     client_hash_id      │     content_hash_id      │ march_impressions │ april_impressions │ declining_label │
│         varchar         │         varchar          │      int128       │      int128       │      int32      │
├─────────────────────────┼──────────────────────────┼───────────────────┼───────────────────┼─────────────────┤
│ client_62f4a7e64f5e0096 │ content_35d979572550dd7f │              1423 │              1180 │               0 │
│ client_62f4a7e64f5e0096 │ content_56a4dabd555eec90 │                 1 │                15 │               0 │
│ client_62f4a7e64f5e0096 │ content_e174e46a5b0733a8 │                 0 │                 0 │               0 │
│ client_62f4a7e64f5e0096 │ content_dc2422b7fc475fd6 │                 0 │                 0 │               0 │
│ client_62f4a7e64f5e0096 │ content_89edcdb8887fe6db │                 0 │                 0 │  

In [22]:
final_data = con.sql("""
WITH march_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.ga4_pageviews,
    f.ga4_sessions,
    f.ga4_engaged_sessions,
    l.declining_label
FROM march_features f
INNER JOIN labeled_data l
USING (client_hash_id, content_hash_id)
""")

final_data

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬───────────────┬──────────────┬──────────────────────┬─────────────────┐
│     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ ga4_pageviews │ ga4_sessions │ ga4_engaged_sessions │ declining_label │
│         varchar         │         varchar          │     int128      │   int128   │    int128     │    int128    │        int128        │      int32      │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼───────────────┼──────────────┼──────────────────────┼─────────────────┤
│ client_62f4a7e64f5e0096 │ content_d0dff76c889de68f │             181 │          0 │          NULL │         NULL │                 NULL │               1 │
│ client_62f4a7e64f5e0096 │ content_67741cce996cfafa │              46 │          1 │          NULL │         NULL │                 NULL │               1 │
│ client_62f4a7e64f5e0096 │ content_2e6360ad20fd7107

In [24]:
clean_data = con.sql("""
SELECT *
FROM final_data
WHERE gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND ga4_pageviews IS NOT NULL
  AND ga4_sessions IS NOT NULL
  AND ga4_engaged_sessions IS NOT NULL
""")

clean_data

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────────┬───────────────┬──────────────┬──────────────────────┬─────────────────┐
│     client_hash_id      │     content_hash_id      │ gsc_impressions │ gsc_clicks │ ga4_pageviews │ ga4_sessions │ ga4_engaged_sessions │ declining_label │
│         varchar         │         varchar          │     int128      │   int128   │    int128     │    int128    │        int128        │      int32      │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────────┼───────────────┼──────────────┼──────────────────────┼─────────────────┤
│ client_9958f0a7ae1df715 │ content_eb0aeedbcfaf2712 │             248 │          0 │             7 │            5 │                    0 │               1 │
│ client_9958f0a7ae1df715 │ content_108500096f9bc481 │             356 │          2 │             9 │            7 │                    1 │               1 │
│ client_9958f0a7ae1df715 │ content_4cec18f637b4c858

FIVE FEATURES


In [26]:
import pandas as pd

ml_data = clean_data.df()

ml_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,declining_label
0,client_73cda7b4e4f265ea,content_9cc72848a8d3fac5,498.0,0.0,0.0,0.0,0.0,1
1,client_73cda7b4e4f265ea,content_a12df7206063c246,10414.0,22.0,3.0,3.0,0.0,1
2,client_73cda7b4e4f265ea,content_4f58b267c88475b4,278.0,0.0,0.0,0.0,0.0,1
3,client_73cda7b4e4f265ea,content_b24b3e38bfd0d66b,3796.0,21.0,13.0,8.0,0.0,0
4,client_73cda7b4e4f265ea,content_da653044a685affa,50.0,1.0,0.0,0.0,0.0,1


In [27]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = ml_data[features]
y = ml_data["declining_label"]

print("Features:", features)
print("Rows:", len(ml_data))

Features: ['gsc_impressions', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions']
Rows: 260736


 LEAKAGE EXPERIMENT

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

honest_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_predictions
)

print("Honest model accuracy:", honest_score)

Training rows: 208588
Test rows: 52148
Honest model accuracy: 0.7934148960650457


I now dd declining_labe as a feature to demonstrate data leakage.

In [31]:
leaky_features = features + ["declining_label"]

X_leaky = ml_data[leaky_features]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
leaky_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

leaky_model.fit(X_train_leaky, y_train_leaky)

leaky_predictions = leaky_model.predict(X_test_leaky)

leaky_score = accuracy_score(
    y_test_leaky,
    leaky_predictions
)

print("Leaky model accuracy:", leaky_score)

Leaky model accuracy: 1.0


SHOW COMPARISON

In [32]:
print("Honest accuracy:", honest_score)
print("Leaky accuracy:", leaky_score)
print("Difference:", leaky_score - honest_score)

Honest accuracy: 0.7934148960650457
Leaky accuracy: 1.0
Difference: 0.20658510393495433


REMOVE LEAK


In [33]:
final_features = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X_honest_final = ml_data[final_features]

print("Final features:")
print(final_features)

Final features:
['gsc_impressions', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions']


 **Leakage lesson**

I deliberately added declining_label to the features. The score became very high because the model was given the answer.

This is data leakage. The label should not be used as a feature.

I removed declining_label and kept only the five safe features.